# Labeling
We label the data collected in [01_data_collection.ipynb](01_data_collection.ipynb) using zero-shot classification via `facebook/bart-large-mnli`.

In [1]:
from transformers import pipeline
import torch

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0,
    dtype=torch.float16,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

We define some `relevant` categories and some `irrelevant` categories.

In [2]:
relevant = {
    "Monetary and fiscal policy",
    "Macroeconomic data and indicators",
    "Geopolitics",
    "International trade",
    "Corporate and sector dynamics",
    "Commodities and supply Chain",
    "Systemic shocks",
    "Natural environment shocks",
    "Public health",
    "Technology advancements",
}

irrelevant = {
    "Entertainment",
    "Culture and art",
    "Sports",
    "Lifestyle, travel, wellness",
    # "Local news",
    "Science and pure research",
    "Long-form Features & Investigative Essays",
    "Personal & Human Interest",
}

all_labels = list(relevant | irrelevant)

We label sequences according to the given relevant/irrelevant classes.

In [3]:
import numpy as np


def relevant_labeler(batch):
    results = classifier(batch["concat"], candidate_labels=all_labels)

    labels = []
    for res in results:
        scores = dict(zip(res["labels"], res["scores"]))

        top_label = res["labels"][0]
        top_score = res["scores"][0]
        rel_sum = sum(scores[cat] for cat in relevant)

        if top_score > 0.5:
            label = 1 if top_label in relevant else 0
        elif rel_sum > 0.65:
            label = 1
        elif rel_sum < 0.35:
            label = 0
        else:
            label = -1

        labels.append(label)

    return {"relevant": labels}

We now load, clean, and label the `raw_headlines.csv` dataset.

We first merge title and description into a HuggingFace `Dataset` for faster processing.

In [4]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("data/raw_headlines.csv")
df["concat"] = df["title"] + " | " + df["description"]
df.dropna(inplace=True)
ds = Dataset.from_pandas(df[["concat"]])

We apply the labeling function: relevant (`1`), irrelevant (`0`), or unknown (`-1`).

In [5]:
ds = ds.map(relevant_labeler, batched=True, batch_size=128)

Map:   0%|          | 0/1323 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


and copy the new columns in the original `pandas` dataframe:

In [8]:
df.dropna(inplace=True)
df["relevant"] = ds["relevant"]

In [9]:
df.to_csv("data/labeled_headlines.csv")

While this is a working pipeline, the labeled categories are hugely unbalanced.  In fact, the RSS sources are already selected for relevance, so that the "irrelevant" category is already naturally small.  To avoid having the student network learn to select only relevant data, we will collect some extra data from irrelevant categories and add it to the training set.